# 🤖 EcoRiego AI — Notebook 2: Entrenamiento con Aprendizaje por Refuerzo
**Proyecto:** EcoRiego AI — Sistema de Riego Autónomo con TinyML para ESP32  
**Algoritmo:** Actor-Critic (REINFORCE with Baseline)

---
### Flujo de este Notebook
```
FASE 1: Entorno de Simulación Agrícola (Gym-like)
    ↓
FASE 2: Sistema de Recompensas Agronómicas (Corazón del RL)
    ↓
FASE 3: Arquitectura Actor-Critic (Red Neuronal escalable)
    ↓
FASE 4: Bucle de Entrenamiento por Refuerzo
    ↓
FASE 5: Validación del Agente
    ↓
FASE 6: Conversión a TFLite → modelo_entrenado.h (ESP32)
```

**¿Por qué RL en lugar de Supervised Learning?**  
El SL aprende a imitar una lógica que ya construimos nosotros. El RL aprende a **maximizar** el resultado agronómico real (humedad óptima + ahorro de agua), encontrando estrategias que la lógica manual nunca hubiera descubierto.


In [ ]:
# !pip install tensorflow pandas numpy matplotlib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import math

print("✅ Librerías cargadas.")
print(f"   TensorFlow v{tf.__version__}  |  NumPy v{np.__version__}")


## FASE 1 — Entorno de Simulación Agrícola

El entorno es un invernadero simplificado que evoluciona hora a hora (24 pasos por episodio).

| Elemento | Descripción |
|----------|-------------|
| **Estado** | `[HumedadSuelo, TemperaturaAire, HumedadAire]` — lo que ve el sensor |
| **Acción** | Tiempo de riego en minutos `[0.0, 15.0]` — lo que decide el agente |
| **Dinámica** | La humedad del suelo sube con el riego y baja con la evapotranspiración |
| **Escalabilidad** | Añade más cultivos en el diccionario `CULTIVOS` sin cambiar el código |


In [ ]:
class EntornoInvernadero:
    """
    Entorno Gym-like de simulación de invernadero para Aprendizaje por Refuerzo.
    Compatible con los 3 sensores del ESP32 (Capacitivo + DHT22).
    Escalable: añade cultivos en CULTIVOS sin modificar el bucle de entrenamiento.
    """

    CULTIVOS = {
        "tomate":  {"optimo_min": 55, "optimo_max": 70, "kc": 1.05},
        "lechuga": {"optimo_min": 60, "optimo_max": 75, "kc": 0.90},
        "papa":    {"optimo_min": 50, "optimo_max": 65, "kc": 1.10},
    }

    def __init__(self, pasos_por_episodio: int = 24, cultivo: str = "tomate"):
        self.pasos_por_episodio = pasos_por_episodio
        self.cultivo  = cultivo
        self.perfil   = self.CULTIVOS[cultivo]
        self.n_obs    = 3   # Número de features (escalable)
        self.paso_actual = 0
        self.estado      = None

    def reset(self) -> np.ndarray:
        self.paso_actual = 0
        self.hora_dia    = np.random.uniform(6, 12)
        self.mes         = np.random.randint(1, 13)
        humedad_suelo    = np.random.uniform(20, 80)
        temperatura      = self._temp(self.hora_dia, self.mes)
        humedad_aire     = self._hum_aire(temperatura, self.mes)
        self.estado = np.array([humedad_suelo, temperatura, humedad_aire], dtype=np.float32)
        return self.estado.copy()

    def step(self, accion_minutos: float):
        accion   = float(np.clip(accion_minutos, 0.0, 15.0))
        suelo, temp, hum_aire = self.estado

        # ── Dinámica del suelo: riego sube, ET baja ──
        et          = self._et(temp, hum_aire, self.hora_dia) * self.perfil["kc"]
        nueva_hum   = suelo + (accion * 1.2) - (et * 8.0)
        nueva_hum   = float(np.clip(nueva_hum, 0.0, 100.0))

        self.hora_dia    = (self.hora_dia + 1) % 24
        self.paso_actual += 1

        nueva_temp   = self._temp(self.hora_dia, self.mes)
        nuevo_haire  = self._hum_aire(nueva_temp, self.mes)

        recompensa, info = self._recompensa(nueva_hum, accion, temp, hum_aire, self.hora_dia)

        self.estado = np.array([nueva_hum, nueva_temp, nuevo_haire], dtype=np.float32)
        terminado   = (self.paso_actual >= self.pasos_por_episodio)

        return self.estado.copy(), recompensa, terminado, info

    def _recompensa(self, hum_suelo, accion, temp, hum_aire, hora):
        """
        Sistema de Recompensas Agronómicas — corazón del RL.
        Diseño: maximizar la planta sana con mínimo uso de agua.
        """
        opt_min = self.perfil["optimo_min"]
        opt_max = self.perfil["optimo_max"]
        r = 0.0; info = {}

        # ✅ R+: Humedad en zona óptima
        if opt_min <= hum_suelo <= opt_max:
            r += 12.0; info["zona"] = "OPTIMA ✅"
        elif hum_suelo < opt_min:
            deficit = opt_min - hum_suelo
            r -= deficit * 0.5; info["zona"] = f"SECO ⚠️ (-{deficit:.1f}%)"
        else:
            exceso = hum_suelo - opt_max
            r -= exceso * 0.3; info["zona"] = f"ENCHARCADO 🚫 (+{exceso:.1f}%)"

        # ❌ P: Regar con suelo ya encharcado
        if accion > 0 and hum_suelo > opt_max:
            r -= accion * 2.0; info["desperdicio"] = "Riego innecesario"

        # ❌ P: Regar de noche (hongos)
        if accion > 0 and (hora < 6 or hora >= 22):
            r -= accion * 1.5; info["noche"] = "Riego nocturno peligroso"

        # ❌ P: Regar con niebla
        if accion > 0 and hum_aire > 85:
            r -= accion * 1.0; info["niebla"] = "Riego con alta humedad ambiental"

        # ✅ R+: Eficiencia (no regar cuando no hace falta)
        if accion == 0 and opt_min <= hum_suelo <= opt_max:
            r += 3.0; info["eficiencia"] = "Ahorro inteligente 💧"

        # ❌ P: Estrés severo
        if hum_suelo < 20: r -= 20.0; info["critico"] = "ESTRÉS HÍDRICO 🚨"
        if hum_suelo > 95: r -= 15.0; info["critico"] = "SATURACIÓN 🚨"

        return float(r), info

    def _temp(self, hora, mes):
        base = 18.0; amp = 10.0
        adj  = 4.0 if 4 <= mes <= 10 else -2.0
        var  = amp * math.sin(math.pi * (hora-6)/12) if 6<=hora<=18 else -amp*0.5
        return float(np.clip(base+adj+var+np.random.normal(0,1.0), 8.0, 38.0))

    def _hum_aire(self, temp, mes):
        base = 75.0 if mes in [11,12,1,2,3] else 55.0
        return float(np.clip(base - 1.5*max(0,temp-20) + np.random.normal(0,4.0), 20.0, 98.0))

    def _et(self, temp, hum_aire, hora):
        fs  = max(0, math.sin(math.pi*(hora-6)/12)) if 6<=hora<=18 else 0.0
        vpd = (1-hum_aire/100)*0.611*math.exp(17.27*temp/(temp+237.3))
        return float(np.clip(0.408*fs*0.5 + 0.066*vpd*2.0, 0.0, 2.5))


# ── Prueba del entorno ──
env_test = EntornoInvernadero(cultivo="tomate")
obs = env_test.reset()
print(f"✅ Entorno creado. Estado inicial:")
print(f"   HumedadSuelo={obs[0]:.1f}%  Temp={obs[1]:.1f}°C  HumAire={obs[2]:.1f}%")
obs2, r, done, info = env_test.step(3.0)
print(f"   Riego: 3.0 min → Recompensa: {r:.2f} | {info}")


## FASE 3 — Arquitectura Actor-Critic

```
ACTOR  (→ ESP32)   CRITIC (solo entrenamiento, se descarta)
   Estado (3)          Estado (3)
      ↓                   ↓
   Dense(32)           Dense(32)
   Dense(16)           Dense(16)
   Dense(8)            Dense(1) → Valor del estado V(s)
   Dense(1)
   Sigmoid×15 → tiempo_minutos [0, 15]
```

**¿Por qué separar Actor y Critic?**  
El Critic actúa como "baseline": en vez de actualizar el Actor con la recompensa cruda (muy ruidosa), usamos la **ventaja** `A = G - V(s)`, que mide si la acción fue *mejor o peor de lo esperado*. Esto reduce la varianza y acelera el aprendizaje.


In [ ]:
N_ENTRADAS = 3  # [HumedadSuelo, Temperatura, HumedadAire] — escalable

def crear_actor() -> keras.Model:
    """
    Red del Actor. ESTA es la que va al ESP32 como modelo_entrenado.h
    Arquitectura: 3 → 32 → 16 → 8 → 1 (escalable añadiendo capas)
    Salida: float en [0, 1] → multiplicar × 15.0 en el firmware para obtener minutos
    """
    entradas = keras.Input(shape=(N_ENTRADAS,), name="sensores")

    # Normalización interna (mejora convergencia)
    # Los pesos de la red aprenden sobre valores en [0,1]
    norm = keras.layers.Lambda(
        lambda x: x / tf.constant([100.0, 40.0, 100.0], dtype=tf.float32),
        name="normalizacion")(entradas)

    x = keras.layers.Dense(32, activation='relu', name="capa_1")(norm)
    x = keras.layers.Dense(16, activation='relu', name="capa_2")(x)
    x = keras.layers.Dense(8,  activation='relu', name="capa_3")(x)

    # Sigmoid: salida [0,1]. En ESP32: tiempo = salida × 15.0
    salida = keras.layers.Dense(1, activation='sigmoid', name="salida_normalizada")(x)

    return keras.Model(inputs=entradas, outputs=salida, name="actor")

def crear_critic() -> keras.Model:
    """Red del Critic. Solo se usa en entrenamiento, NO va al ESP32."""
    entradas = keras.Input(shape=(N_ENTRADAS,), name="estado")
    x = keras.layers.Dense(32, activation='relu')(entradas)
    x = keras.layers.Dense(16, activation='relu')(x)
    valor = keras.layers.Dense(1, name="valor")(x)
    return keras.Model(inputs=entradas, outputs=valor, name="critic")

actor  = crear_actor()
critic = crear_critic()

actor.summary()
print(f"\n  Parámetros del Actor  (→ ESP32) : {actor.count_params():,}")
print(f"  Parámetros del Critic (descarta): {critic.count_params():,}")


## FASE 4 — Bucle de Entrenamiento por Refuerzo

### Algoritmo: Actor-Critic REINFORCE with Baseline

**Por episodio:**
1. El agente recorre 24 horas del invernadero explorando con ruido
2. Recolecta trayectoria: `(estado, acción, recompensa)` × 24
3. Calcula retornos descontados: `G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ...`
4. Actualiza Critic: minimiza `(G_t - V(s_t))²`
5. Actualiza Actor: minimiza error de acción ponderado por ventaja `A_t = G_t - V(s_t)`
6. El ruido de exploración decae gradualmente (más explotación con el tiempo)


In [ ]:
# ── Hiperparámetros ──────────────────────────────────────────────
LR_ACTOR       = 3e-4   # Tasa de aprendizaje del Actor
LR_CRITIC      = 1e-3   # Tasa de aprendizaje del Critic
GAMMA          = 0.95   # Factor de descuento temporal
N_EPISODIOS    = 600    # Número de episodios de entrenamiento
PASOS_EPISODIO = 24     # 24 horas simuladas por episodio
RUIDO_INICIAL  = 1.5    # Exploración inicial (ruido gaussiano en acciones)
RUIDO_MINIMO   = 0.05   # Mínimo ruido (fase de explotación)
DECAIMIENTO    = 0.995  # Factor de reducción del ruido por episodio
# ─────────────────────────────────────────────────────────────────

opt_actor  = keras.optimizers.Adam(learning_rate=LR_ACTOR)
opt_critic = keras.optimizers.Adam(learning_rate=LR_CRITIC)

hist_recompensas = []
hist_actor_loss  = []
hist_critic_loss = []

env   = EntornoInvernadero(pasos_por_episodio=PASOS_EPISODIO, cultivo="tomate")
ruido = RUIDO_INICIAL
mejor_media = -9999.0

print("=" * 65)
print("  ENTRENAMIENTO ACTOR-CRITIC — EcoRiego AI")
print("=" * 65)
print(f"  Episodios  : {N_EPISODIOS}  |  Pasos/Ep : {PASOS_EPISODIO}h simuladas")
print(f"  LR Actor   : {LR_ACTOR}  |  LR Critic: {LR_CRITIC}")
print(f"  γ (gamma)  : {GAMMA}  |  Ruido↓ : {RUIDO_INICIAL}→{RUIDO_MINIMO}")
print("=" * 65)

for ep in range(1, N_EPISODIOS + 1):
    estado = env.reset()
    buf_estados, buf_acciones, buf_rewards = [], [], []

    # ── Recolección de trayectoria ──
    for _ in range(PASOS_EPISODIO):
        s_tensor = tf.constant([estado], dtype=tf.float32)
        # Actor predice [0,1], escalamos a [0,15] para el entorno
        a_norm   = float(actor(s_tensor, training=False).numpy()[0, 0])
        a_minutos = a_norm * 15.0
        # Exploración: ruido gaussiano en espacio de minutos
        a_explorar = float(np.clip(a_minutos + np.random.normal(0, ruido), 0.0, 15.0))

        nuevo_estado, recompensa, done, _ = env.step(a_explorar)

        buf_estados.append(estado.copy())
        buf_acciones.append(a_explorar / 15.0)  # Guardar normalizado [0,1]
        buf_rewards.append(recompensa)

        estado = nuevo_estado
        if done:
            break

    T = len(buf_rewards)

    # ── Calcular retornos descontados ──
    G = np.zeros(T, dtype=np.float32)
    acum = 0.0
    for t in reversed(range(T)):
        acum = buf_rewards[t] + GAMMA * acum
        G[t] = acum

    # Normalizar retornos (estabiliza el gradiente)
    G = (G - G.mean()) / (G.std() + 1e-8)

    s_t  = tf.constant(buf_estados, dtype=tf.float32)
    a_t  = tf.constant([[a] for a in buf_acciones], dtype=tf.float32)
    G_t  = tf.constant(G.reshape(-1, 1), dtype=tf.float32)

    # ── Actualizar Critic ──
    with tf.GradientTape() as tc:
        V = critic(s_t, training=True)
        loss_c = tf.reduce_mean(tf.square(G_t - V))
    grads_c = tc.gradient(loss_c, critic.trainable_variables)
    opt_critic.apply_gradients(zip(grads_c, critic.trainable_variables))

    # ── Actualizar Actor (ponderado por ventaja A = G - V) ──
    with tf.GradientTape() as ta:
        a_pred   = actor(s_t, training=True)
        ventaja  = G_t - tf.stop_gradient(critic(s_t))
        loss_a   = tf.reduce_mean(tf.square(a_t - a_pred) * tf.stop_gradient(ventaja))
    grads_a = ta.gradient(loss_a, actor.trainable_variables)
    opt_actor.apply_gradients(zip(grads_a, actor.trainable_variables))

    ruido = max(RUIDO_MINIMO, ruido * DECAIMIENTO)

    recompensa_ep = sum(buf_rewards)
    hist_recompensas.append(recompensa_ep)
    hist_actor_loss.append(float(loss_a.numpy()))
    hist_critic_loss.append(float(loss_c.numpy()))

    if ep % 60 == 0:
        media = np.mean(hist_recompensas[-100:])
        print(f"  Ep {ep:4d} | Recompensa: {recompensa_ep:7.1f} | Media(100): {media:7.1f} | Ruido: {ruido:.3f}")
        if media > mejor_media:
            mejor_media = media
            actor.save_weights("/tmp/actor_mejor.weights.h5")

print(f"\n✅ Entrenamiento finalizado. Mejor media: {mejor_media:.1f}")
actor.load_weights("/tmp/actor_mejor.weights.h5")
print("   ✅ Mejor actor cargado.")


## FASE 5 — Validación del Agente

In [ ]:
print("=" * 70)
print("  VALIDACIÓN — Tabla de Casos (comparable con Tabla 7 del informe)")
print("=" * 70)

casos = [
    {"nombre": "Sequía Total",      "suelo": 0,  "temp": 24, "hum_aire": 55},
    {"nombre": "Suelo Seco",        "suelo": 33, "temp": 24, "hum_aire": 55},
    {"nombre": "Suelo Húmedo",      "suelo": 51, "temp": 24, "hum_aire": 60},
    {"nombre": "Seco + Frío",       "suelo": 30, "temp": 10, "hum_aire": 80},
    {"nombre": "Óptimo + Niebla",   "suelo": 65, "temp": 15, "hum_aire": 92},
    {"nombre": "Saturado + Calor",  "suelo": 90, "temp": 32, "hum_aire": 35},
    {"nombre": "Crítico: estrés",   "suelo": 10, "temp": 34, "hum_aire": 25},
    {"nombre": "Nocturno (3am)",    "suelo": 40, "temp": 14, "hum_aire": 80},
]

print(f"  {'Caso':<23} {'Suelo':>6} {'Temp':>6} {'HumA':>6} │ {'Acción':>8}  Interpretación")
print("  " + "─"*78)

for c in casos:
    inp    = np.array([[c["suelo"], c["temp"], c["hum_aire"]]], dtype=np.float32)
    salida = float(actor(inp, training=False).numpy()[0,0])
    tiempo = round(float(np.clip(salida * 15.0, 0, 15)), 2)

    if tiempo < 0.5:   interp = "🚫 No regar"
    elif tiempo < 4.0: interp = "💧 Riego suave"
    elif tiempo < 8.0: interp = "💦 Riego moderado"
    else:              interp = "🌊 Riego intenso"

    print(f"  {c['nombre']:<23} {c['suelo']:>5}% {c['temp']:>5}°C {c['hum_aire']:>5}% │ {tiempo:>7.2f}m  {interp}")

# ── Curvas de aprendizaje ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("EcoRiego AI — Curvas de Aprendizaje Actor-Critic", fontsize=12, fontweight='bold')

def mm(data, w=30):
    return pd.Series(data).rolling(w, min_periods=1).mean().values

eps = range(1, N_EPISODIOS + 1)
colores = ["#2196F3", "#FF9800", "#9C27B0"]
titulos = ["Recompensa por Episodio", "Loss del Actor", "Loss del Critic"]
datos   = [hist_recompensas, hist_actor_loss, hist_critic_loss]

for ax, d, c, t in zip(axes, datos, colores, titulos):
    ax.plot(eps, d, alpha=0.25, color=c, linewidth=0.7)
    ax.plot(eps, mm(d), color=c, linewidth=2)
    ax.set_title(t); ax.set_xlabel("Episodio"); ax.grid(alpha=0.3)

axes[0].axhline(0, color='red', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig("curvas_entrenamiento.png", dpi=120, bbox_inches='tight')
plt.show()
print("📊 Curvas guardadas: curvas_entrenamiento.png")


## FASE 6 — Conversión a TFLite → `modelo_entrenado.h`

Este es el paso final: convertir el Actor entrenado en un archivo `.h` que se incluye en el firmware del ESP32.

```cpp
// En IrrigationBrain.cpp — uso del modelo:
float input[3] = {
    humedad_suelo / 100.0f,  // Normalizar igual que en entrenamiento
    temperatura   / 40.0f,
    humedad_aire  / 100.0f
};
float salida = inferencia(input);         // Salida del modelo: [0, 1]
float tiempo_min = salida * 15.0f;        // Convertir a minutos
```


In [ ]:
# ── Paso 1: Guardar modelo Keras ──
actor.save("actor_ecoriego.keras")
print("✅ Modelo Keras guardado: actor_ecoriego.keras")

# ── Paso 2: Reconstruir sin capas Lambda (mejor compatibilidad TFLite Micro) ──
def crear_actor_tflite() -> keras.Model:
    """
    Versión del Actor sin capas Lambda para máxima compatibilidad con TFLite Micro.
    Entrada: [HumedadSuelo/100, Temperatura/40, HumedadAire/100] — ya normalizados
    Salida : float [0,1] → multiplicar × 15.0 en firmware para obtener minutos
    """
    entradas = keras.Input(shape=(3,), name="sensores_norm")
    x = keras.layers.Dense(32, activation='relu', name="d1")(entradas)
    x = keras.layers.Dense(16, activation='relu', name="d2")(x)
    x = keras.layers.Dense(8,  activation='relu', name="d3")(x)
    salida = keras.layers.Dense(1, activation='sigmoid', name="tiempo_norm")(x)
    return keras.Model(inputs=entradas, outputs=salida, name="actor_tflite")

actor_export = crear_actor_tflite()

# Transferir pesos de capas Dense (misma arquitectura)
capas_src = [l for l in actor.layers if isinstance(l, keras.layers.Dense)]
capas_dst = [l for l in actor_export.layers if isinstance(l, keras.layers.Dense)]
for s, d in zip(capas_src, capas_dst):
    d.set_weights(s.get_weights())

print("✅ Modelo TFLite-compatible construido con pesos transferidos.")

# ── Paso 3: Conversión a TFLite con cuantización ──
converter = tf.lite.TFLiteConverter.from_keras_model(actor_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Datos representativos para calibrar cuantización
def datos_rep():
    for _ in range(300):
        yield [np.array([[
            np.random.uniform(0,1),    # HumedadSuelo normalizada
            np.random.uniform(0,1),    # Temperatura normalizada
            np.random.uniform(0,1),    # HumedadAire normalizada
        ]], dtype=np.float32)]

converter.representative_dataset = datos_rep

try:
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8
    tflite_model = converter.convert()
    print(f"✅ Cuantización INT8 exitosa. Tamaño: {len(tflite_model)/1024:.1f} KB")
except Exception as e:
    print(f"⚠️ INT8 no disponible ({e}). Usando DEFAULT...")
    converter2 = tf.lite.TFLiteConverter.from_keras_model(actor_export)
    converter2.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter2.convert()
    print(f"✅ Cuantización DEFAULT. Tamaño: {len(tflite_model)/1024:.1f} KB")

# Guardar .tflite
with open("riego_model.tflite", "wb") as f:
    f.write(tflite_model)
print("✅ Archivo 'riego_model.tflite' guardado.")

# ── Paso 4: Generar modelo_entrenado.h ──
def bytes_a_header(data: bytes, nombre: str = "riego_model") -> str:
    lines = [
        "// ══════════════════════════════════════════════════════════",
        "// EcoRiego AI — Modelo entrenado con Aprendizaje por Refuerzo",
        "// Generado automáticamente. No editar manualmente.",
        f"// Tamaño   : {len(data)} bytes  ({len(data)/1024:.1f} KB)",
        "// Entradas  : [HumedadSuelo/100.0, Temperatura/40.0, HumedadAire/100.0]",
        "// Salida    : float [0,1]  →  tiempo_min = salida × 15.0f",
        "// Algoritmo : Actor-Critic REINFORCE with Baseline",
        "// ══════════════════════════════════════════════════════════",
        "", "#pragma once", "#include <cstdint>", "",
        f"alignas(8) const unsigned char {nombre}[] = {{"
    ]
    hex_vals = [f"0x{b:02x}" for b in data]
    for i in range(0, len(hex_vals), 16):
        lines.append("  " + ", ".join(hex_vals[i:i+16]) + ",")
    lines += ["};", "", f"const unsigned int {nombre}_len = {len(data)};", ""]
    return "\n".join(lines)

header = bytes_a_header(tflite_model)
with open("modelo_entrenado.h", "w") as f:
    f.write(header)
print("✅ Archivo 'modelo_entrenado.h' generado.")


In [ ]:
print("\n" + "=" * 65)
print("  ✅ PIPELINE COMPLETO — EcoRiego AI  RL Edition")
print("=" * 65)
print()
print("  ARCHIVOS PARA EL ESP32:")
print(f"    📄 modelo_entrenado.h       → Incluir en IrrigationBrain.h")
print(f"    📄 riego_model.tflite       → {len(tflite_model)/1024:.1f} KB")
print(f"    📄 actor_ecoriego.keras     → Backup del modelo completo")
print(f"    📊 curvas_entrenamiento.png")
print()
print("  INSTRUCCIÓN CLAVE EN IrrigationBrain.cpp:")
print("  ─────────────────────────────────────────")
print("  // IMPORTANTE: Normalizar las entradas antes de inferencia")
print("  float input[3] = {")
print("    humedad_suelo / 100.0f,   // Sensor capacitivo [0,100] → [0,1]")
print("    temperatura   / 40.0f,    // DHT22 [0,40°C] → [0,1]")
print("    humedad_aire  / 100.0f    // DHT22 [0,100%] → [0,1]")
print("  };")
print("  float salida = tflite_infer(input);    // salida en [0,1]")
print("  float tiempo_minutos = salida * 15.0f; // Escalar a [0,15] min")
print()
print("  MEJORAS VS VERSIÓN ANTERIOR (Supervised Learning):")
print("  ─────────────────────────────────────────────────")
print("  ✅ Aprende por recompensas reales (no imita reglas manuales)")
print("  ✅ Sistema de recompensas agronómicas (ET, niebla, noche, fase)")
print("  ✅ Entorno dinámico con física de invernadero real")
print("  ✅ Arquitectura Actor-Critic escalable")
print("  ✅ Cuantización INT8 → menor tamaño en flash del ESP32")
print("=" * 65)
